# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [18]:
import pandas as pd

chunks = pd.read_json("data/raw/Books_5.json", lines=True, chunksize=100000)
first_chunk = next(chunks)
print(len(first_chunk))

# Just for testing
first_chunk.head()


100000


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A10000012B7CGYKOMPQ4L,000100039X,Adam,"[0, 0]",Spiritually and mentally inspiring! A book tha...,5,Wonderful!,1355616000,"12 16, 2012"
1,A2S166WSCFIFP5,000100039X,"adead_poet@hotmail.com ""adead_poet@hotmail.com""","[0, 2]",This is one my must have books. It is a master...,5,close to god,1071100800,"12 11, 2003"
2,A1BM81XB4QHOA3,000100039X,"Ahoro Blethends ""Seriously""","[0, 0]",This book provides a reflection that you can a...,5,Must Read for Life Afficianados,1390003200,"01 18, 2014"
3,A1MOSTXNIO5MPJ,000100039X,Alan Krug,"[0, 0]",I first read THE PROPHET in college back in th...,5,Timeless for every good and bad time in your l...,1317081600,"09 27, 2011"
4,A2XQ5LZHTD4AFT,000100039X,Alaturka,"[7, 9]",A timeless classic. It is a very demanding an...,5,A Modern Rumi,1033948800,"10 7, 2002"


In [21]:
df = first_chunk
df["user_id"] = pd.factorize(df["reviewerID"])[0]
df["item_id"] = pd.factorize(df["asin"])[0]
df["interaction"] = 1


In [23]:
df = df.sort_values(["user_id", "unixReviewTime"]).copy()

df_test = df.groupby("user_id").tail(1).copy()
df_train = df.drop(df_test.index).copy()

df_train = df_train.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

# MostPopular section

In [27]:
item_popularity = (
    df_train.groupby("item_id")
    .size()
    .sort_values(ascending=False)
)

popular_items = item_popularity.index.tolist()
user_seen_train = df_train.groupby("user_id")["item_id"].apply(set).to_dict()

print(popular_items)

[591, 11, 552, 601, 1552, 244, 1873, 593, 42, 1684, 2040, 172, 321, 367, 1310, 315, 758, 2607, 814, 2447, 391, 2135, 817, 2146, 1693, 195, 280, 2351, 13, 2328, 757, 691, 260, 2011, 373, 274, 292, 2005, 1779, 386, 2403, 2587, 560, 2038, 190, 2603, 2421, 2169, 2103, 1871, 320, 1152, 457, 2151, 2393, 1863, 136, 2064, 1461, 1459, 2520, 521, 587, 0, 1460, 1472, 1643, 38, 39, 293, 2614, 437, 1838, 1228, 2525, 1416, 2375, 1833, 477, 1327, 441, 294, 1981, 1368, 2524, 2012, 2522, 2439, 276, 142, 1158, 2523, 2554, 2440, 1205, 1988, 509, 159, 180, 1692, 277, 1581, 2506, 2254, 2157, 249, 819, 1332, 165, 2102, 2325, 574, 1900, 1058, 2615, 1178, 1995, 400, 2108, 1880, 575, 1359, 219, 878, 833, 2019, 2576, 325, 1296, 2109, 1525, 364, 1153, 2162, 2359, 2058, 553, 455, 1832, 1443, 1361, 160, 1388, 1093, 635, 1331, 2153, 2118, 2178, 1805, 1477, 1336, 2635, 2110, 446, 745, 415, 576, 1656, 2211, 716, 1269, 69, 1257, 2507, 2424, 1734, 569, 409, 241, 1735, 2304, 1284, 1865, 2390, 2385, 568, 1342, 2405, 1732

In [30]:
import math


def recommend_most_popular(user_id, popular_items, user_seen_train, k=10):
    seen = user_seen_train.get(user_id, set())
    recs = []
    for item in popular_items:
        if item not in seen:
            recs.append(item)
        if len(recs) == k:
            break
    return recs

def recall_at_k(recommended_items, ground_truth_item):
    return 1.0 if ground_truth_item in recommended_items else 0.0

def ndcg_at_k(recommended_items, ground_truth_item):
    if ground_truth_item in recommended_items:
        rank = recommended_items.index(ground_truth_item) + 1
        return 1.0 / math.log2(rank + 1)
    return 0.0

In [31]:
K = 10
recalls = []
ndcgs = []

for _, row in df_test.iterrows():
    user_id = row["user_id"]
    ground_truth_item = row["item_id"]

    recs = recommend_most_popular(user_id, popular_items, user_seen_train, k=K)

    recalls.append(recall_at_k(recs, ground_truth_item))
    ndcgs.append(ndcg_at_k(recs, ground_truth_item))

print(f"Users evaluated: {len(recalls)}")
# reccomend 10 items and see in what percentage of users that item is in the top 10.
print(f"Recall@{K}: {sum(recalls)/len(recalls):.4f}")
# same but looks at ranking qaulity
print(f"NDCG@{K}: {sum(ndcgs)/len(ndcgs):.4f}")

Users evaluated: 68136
Recall@10: 0.2463
NDCG@10: 0.1230


## BRP

Train shape: (31864, 12)
Test shape: (14778, 12)
Users: 14975 Items: 2508


100%|██████████| 100/100 [00:01<00:00, 72.04it/s, train_auc=99.92%, skipped=1.72%]


IndexError: index 2508 is out of bounds for axis 0 with size 2508